# How to create a sensor

This tutorial demonstrates how to create a sensor.

There are different type of sensors available: irradiance sensor, radiance sensor, camera sensor.

## Prerequisites

### Perform imports

In [1]:
from pathlib import Path

In [2]:
from ansys.speos.core import Face, Project, Speos, launcher
from ansys.speos.core.generic.parameters import (
    CameraSensorParameters,
    ColorParameters,
    EnergyDensity3DDimensionsParameters,
    EnergyDensity3DSensorParameters,
    HumanEyeSensorParameters,
    HumanEyeSensorTypes,
    LayerTypes,
    ObserverSensorParameters,
    PhotometricCameraParameters,
    PolarIntensityDimensionsParameters,
    PolarIntensityFormatTypes,
    PolarIntensitySensorParameters,
    SensorTypes,
)
from ansys.speos.core.generic.version_checker import server_version_checker
from ansys.speos.core.kernel.client import (
    default_docker_channel,
)
from ansys.speos.core.sensor import (
    Sensor3DEnergyDensity,
    Sensor3DIrradiance,
    SensorCamera,
    SensorHumanEye,
    SensorImmersive,
    SensorIrradiance,
    SensorObserver,
    SensorPolarIntensity,
    SensorRadiance,
)

### Define constants

The constants help ensure consistency and avoid repetition throughout the example.

In [3]:
HOSTNAME = "localhost"
GRPC_PORT = 50098  # Be sure the Speos GRPC Server has been started on this port.
USE_DOCKER = True  # Set to False if you're running this example locally as a Notebook.
FILES = "CameraInputFiles"

### Define helper functions

In [4]:
def create_helper_geometries(project: Project):
    """Create bodies and faces."""

    def create_face(body):
        f = body.create_face(name="TheFaceF")
        f.vertices = [0, 0, 0, 1, 0, 0, 0, 1, 0]
        f.facets = [0, 1, 2]
        f.normals = [0, 0, 1, 0, 0, 1, 0, 0, 1]
        f.commit()

    root_part = project.create_root_part().commit()
    body_b1 = root_part.create_body(name="TheBodyB").commit()
    body_b2 = root_part.create_body(name="TheBodyC").commit()
    body_b3 = root_part.create_body(name="TheBodyD").commit()
    body_b4 = root_part.create_body(name="TheBodyE").commit()
    for b in [body_b1, body_b2, body_b3, body_b4]:
        create_face(b)

## Model Setup

### Load assets
The assets used to run this example are available in the
[PySpeos repository](https://github.com/ansys/pyspeos/) on GitHub.

> **Note:** Make sure you
> have downloaded simulation assets and set ``assets_data_path``
> to point to the assets folder.

In [5]:
if USE_DOCKER:  # Running on the remote server.
    assets_data_path = Path("/app") / "assets"
else:
    assets_data_path = Path("/path/to/your/download/assets/directory")

### Connect to the RPC Server
This Python client connects to a server where the Speos engine
is running as a service. In this example, the server and
client are the same machine. The launch_local_speos_rpc_method can
be used to start a local instance of the service.

In [6]:
if USE_DOCKER:
    speos = Speos(channel=default_docker_channel())
else:
    speos = launcher.launch_local_speos_rpc_server(port=GRPC_PORT)

/home/runner/work/pyspeos/pyspeos/.venv/lib/python3.14/site-packages/ansys/tools/common/cyberchannel.py:201: UserWarning: Starting gRPC client without TLS on localhost:50098. This is INSECURE. Consider using a secure connection.
  warn(f"Starting gRPC client without TLS on {target}. This is INSECURE. Consider using a secure connection.")


## Create a new project

The only way to create a sensor, is to create it from a project.

In [7]:
p = Project(speos=speos)
print(p)

{
    "name": "",
    "description": "",
    "metadata": {},
    "part_guid": "",
    "sub_scene_anchor_axis_system": [],
    "sources": [],
    "sensors": [],
    "simulations": [],
    "materials": [],
    "scenes": [],
    "textures_3d": []
}


## Create

Create locally.

The mention "local: " is added when printing the sensor

In [8]:

distortion_file_path = str(assets_data_path / FILES / "CameraDistortion_130deg.OPTDistortion")
transmittance_file_path = str(assets_data_path / FILES / "CameraTransmittance.spectrum")
blue_spectrum_path = str(assets_data_path / FILES / "CameraSensitivityBlue.spectrum")
green_spectrum_path = str(assets_data_path / FILES / "CameraSensitivityGreen.spectrum")
red_spectrum_path = str(assets_data_path / FILES / "CameraSensitivityRed.spectrum")

sensor1 = p.create_sensor(name="Camera.1", feature_type=SensorCamera)
sensor1.distortion_file_uri = distortion_file_path
# Choose photometric mode
sensor1.set_mode_photometric().transmittance_file_uri = transmittance_file_path
# Choose color mode (will imply to give spectrum file for blue, green and red)
mode_color = sensor1.photometric.set_mode_color()
mode_color.blue_spectrum_file_uri = blue_spectrum_path
mode_color.green_spectrum_file_uri = green_spectrum_path
mode_color.red_spectrum_file_uri = red_spectrum_path
print(sensor1)

local: {
    "name": "Camera.1",
    "display_name": "",
    "description": "",
    "metadata": {},
    "sensor_guid": "",
    "result_file_name": "",
    "sensor": {
        "name": "Camera.1",
        "camera": {
            "mode_photometric": {
                "acquisition_integration": 0.01,
                "gamma_correction": 2.2,
                "png_bits": "PNG_BITS_16",
                "mode_color": {
                    "white_balance_mode_none": {},
                    "red_spectrum_guid": "",
                    "green_spectrum_guid": "",
                    "blue_spectrum_guid": ""
                },
                "wavelengths_range": {
                    "w_start": 400.0,
                    "w_end": 700.0,
                    "w_sampling": 13
                },
                "consider_diffraction_effects": false,
                "acquisition_lag_time": 0.0,
                "transmittance_spectrum_guid": ""
            },
            "distortion_file_uri": "/app/asse

## Push it to the server.

Now that it is committed to the server, the mention "local: " is no more present when printing the
sensor.

In [9]:
sensor1.commit()
print(sensor1)

{
    "name": "Camera.1",
    "metadata": {
        "UniqueId": "e4e4ecfb-d368-4ff4-a35b-2e0a6f9b2ee8"
    },
    "sensor_guid": "e4a269f9-8c2c-4694-acd1-53cf14861dd4",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "Camera.1",
        "camera": {
            "mode_photometric": {
                "acquisition_integration": 0.01,
                "transmittance_spectrum_guid": "1bedf5fd-60d5-40a3-8364-38829c318886",
                "gamma_correction": 2.2,
                "png_bits": "PNG_BITS_16",
                "mode_color": {
                    "red_spectrum_guid": "304727c5-1e33-4f06-b06c-39b65b20fd26",
                    "green_spectrum_guid": "d6024651-281a-4da7-abba-47ea5d9d89ab",
                    "blue_spectrum_guid": "426b0880-ceff-4241-8a59-67cdf4a1c2be",
                    "white_balance_mode_none": {},
                    "red_spectrum": {
                        "name": "Camera.1.RedSpectrum",
               

## Another example

Set more characteristics, and use parameters classes.

Camera feature is created with the same default values as the GUI speos.

If the user would like to modify the camera characteristics,
it is possible to do so as below.

In [10]:
distortion_file_path = str(assets_data_path / FILES / "CameraDistortion_130deg.OPTDistortion")
transmittance_file_path = str(assets_data_path / FILES / "CameraTransmittance.spectrum")
blue_spectrum_path = str(assets_data_path / FILES / "CameraSensitivityBlue.spectrum")
green_spectrum_path = str(assets_data_path / FILES / "CameraSensitivityGreen.spectrum")
red_spectrum_path = str(assets_data_path / FILES / "CameraSensitivityRed.spectrum")

color_params = ColorParameters(
    blue_spectrum_file_uri=blue_spectrum_path,
    green_spectrum_file_uri=green_spectrum_path,
    red_spectrum_file_uri=red_spectrum_path,
)
photo_params = PhotometricCameraParameters(
    color_mode=color_params, layer_type="by_source", transmittance_file_uri=transmittance_file_path
)
param = CameraSensorParameters(
    sensor_type_parameters=photo_params,
    distortion_file_uri=distortion_file_path,
    axis_system=[20, 10, 40, 1, 0, 0, 0, 1, 0, 0, 0, 1],
    # camera location [Origin, Xvector, Yvector, Zvector]
    focal_length=5.5,
    height=6,  # dimensions
    width=6,
)

sensor2 = p.create_sensor("Camera_Parameter", feature_type=SensorCamera, parameters=param)
sensor2.commit()

print(sensor2)

{
    "name": "Camera_Parameter",
    "metadata": {
        "UniqueId": "89284e17-2542-4900-a8e1-343266f49c5e"
    },
    "sensor_guid": "78204dad-0cde-4f25-a40d-31312272491a",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "Camera_Parameter",
        "camera": {
            "mode_photometric": {
                "acquisition_integration": 0.01,
                "transmittance_spectrum_guid": "73b942f2-6504-486b-aa33-e8dd25f6ad26",
                "gamma_correction": 2.2,
                "png_bits": "PNG_BITS_16",
                "mode_color": {
                    "red_spectrum_guid": "dab54aa6-96dd-4363-88a9-86db146d18a6",
                    "green_spectrum_guid": "257acc20-983c-4dbb-a492-8aca3830e675",
                    "blue_spectrum_guid": "fd94e77b-b4d6-4c47-8a94-e385022eae78",
                    "white_balance_mode_none": {},
                    "red_spectrum": {
                        "name": "Camera_Parameter.RedSp

## Read

### Sensor Instance

A mention "local: " is added if it is not yet committed to the server

In [11]:
print(sensor1)

{
    "name": "Camera.1",
    "metadata": {
        "UniqueId": "e4e4ecfb-d368-4ff4-a35b-2e0a6f9b2ee8"
    },
    "sensor_guid": "e4a269f9-8c2c-4694-acd1-53cf14861dd4",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "Camera.1",
        "camera": {
            "mode_photometric": {
                "acquisition_integration": 0.01,
                "transmittance_spectrum_guid": "1bedf5fd-60d5-40a3-8364-38829c318886",
                "gamma_correction": 2.2,
                "png_bits": "PNG_BITS_16",
                "mode_color": {
                    "red_spectrum_guid": "304727c5-1e33-4f06-b06c-39b65b20fd26",
                    "green_spectrum_guid": "d6024651-281a-4da7-abba-47ea5d9d89ab",
                    "blue_spectrum_guid": "426b0880-ceff-4241-8a59-67cdf4a1c2be",
                    "white_balance_mode_none": {},
                    "red_spectrum": {
                        "name": "Camera.1.RedSpectrum",
               

### Project

Committed feature will appear in the project.

In [12]:
print(p)

{
    "sensors": [
        {
            "name": "Camera.1",
            "metadata": {
                "UniqueId": "e4e4ecfb-d368-4ff4-a35b-2e0a6f9b2ee8"
            },
            "sensor_guid": "e4a269f9-8c2c-4694-acd1-53cf14861dd4",
            "display_name": "",
            "description": "",
            "result_file_name": "",
            "sensor": {
                "name": "Camera.1",
                "camera": {
                    "mode_photometric": {
                        "acquisition_integration": 0.01,
                        "transmittance_spectrum_guid": "1bedf5fd-60d5-40a3-8364-38829c318886",
                        "gamma_correction": 2.2,
                        "png_bits": "PNG_BITS_16",
                        "mode_color": {
                            "red_spectrum_guid": "304727c5-1e33-4f06-b06c-39b65b20fd26",
                            "green_spectrum_guid": "d6024651-281a-4da7-abba-47ea5d9d89ab",
                            "blue_spectrum_guid": "426b0880-cef

## Update

Tipp: if you are manipulating a sensor already committed, don't forget to commit your changes.

If you don't, you will still only watch what is committed on the server.

In [13]:
# modify f number and axis system
sensor1.f_number = 11
sensor1.axis_system = [17, 10, 10, 1, 0, 0, 0, 1, 0, 0, 0, 1]
sensor1.commit()
print(sensor1)

{
    "name": "Camera.1",
    "metadata": {
        "UniqueId": "e4e4ecfb-d368-4ff4-a35b-2e0a6f9b2ee8"
    },
    "sensor_guid": "e4a269f9-8c2c-4694-acd1-53cf14861dd4",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "Camera.1",
        "camera": {
            "mode_photometric": {
                "acquisition_integration": 0.01,
                "transmittance_spectrum_guid": "1bedf5fd-60d5-40a3-8364-38829c318886",
                "gamma_correction": 2.2,
                "png_bits": "PNG_BITS_16",
                "mode_color": {
                    "red_spectrum_guid": "304727c5-1e33-4f06-b06c-39b65b20fd26",
                    "green_spectrum_guid": "d6024651-281a-4da7-abba-47ea5d9d89ab",
                    "blue_spectrum_guid": "426b0880-ceff-4241-8a59-67cdf4a1c2be",
                    "white_balance_mode_none": {},
                    "red_spectrum": {
                        "name": "Camera.1.RedSpectrum",
               

In [14]:
# Modify photometric camera properties (including diffraction effects)
sensor1.distortion_file_uri = (
    assets_data_path / "CameraInputFiles" / "diffractive_effects.OPTDistortion"
)
sensor1.set_mode_photometric().set_mode_color().red_spectrum_file_uri = str(
    assets_data_path / "CameraInputFiles" / "CameraSensitivityRed.spectrum"
)
sensor1.set_mode_photometric().set_mode_color().green_spectrum_file_uri = str(
    assets_data_path / "CameraInputFiles" / "CameraSensitivityGreen.spectrum"
)
sensor1.set_mode_photometric().set_mode_color().blue_spectrum_file_uri = str(
    assets_data_path / "CameraInputFiles" / "CameraSensitivityBlue.spectrum"
)
wl = sensor1.set_mode_photometric().set_wavelengths_range()
wl.start = 486
wl.end = 655
wl.sampling = 13
sensor1.set_mode_photometric().consider_diffraction_effects = True
sensor1.commit()
print(sensor1)

{
    "name": "Camera.1",
    "metadata": {
        "UniqueId": "e4e4ecfb-d368-4ff4-a35b-2e0a6f9b2ee8"
    },
    "sensor_guid": "e4a269f9-8c2c-4694-acd1-53cf14861dd4",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "Camera.1",
        "camera": {
            "mode_photometric": {
                "acquisition_integration": 0.01,
                "gamma_correction": 2.2,
                "png_bits": "PNG_BITS_16",
                "mode_color": {
                    "red_spectrum_guid": "304727c5-1e33-4f06-b06c-39b65b20fd26",
                    "green_spectrum_guid": "d6024651-281a-4da7-abba-47ea5d9d89ab",
                    "blue_spectrum_guid": "426b0880-ceff-4241-8a59-67cdf4a1c2be",
                    "white_balance_mode_none": {},
                    "red_spectrum": {
                        "name": "Camera.1.RedSpectrum",
                        "library": {
                            "file_uri": "/app/assets/CameraInputF

/home/runner/work/pyspeos/pyspeos/src/ansys/speos/core/sensor.py:1414: UserWarning: Please note that the following values ['focal_length', 'imager_distance', 'f_number', 'Transmittance Spectrum'] were over written by the values stored in the distortion file
  self._commit_template()


## Reset

Possibility to reset local values from the one available in the server.

In [15]:
sensor1.set_mode_geometric()  # set camera in geometric mode but no commit
sensor1.reset()  # reset -> this will apply the server value (photometric mode) to the local value
sensor1.delete()  # delete (to display the local value with the below print)
print(sensor1._sensor_template)

ansys.api.speos.sensor.v2.SensorTemplate
{
    "name": "Camera.1",
    "camera": {
        "mode_photometric": {
            "acquisition_integration": 0.01,
            "gamma_correction": 2.2,
            "png_bits": "PNG_BITS_16",
            "mode_color": {
                "white_balance_mode_none": {},
                "red_spectrum_guid": "",
                "green_spectrum_guid": "",
                "blue_spectrum_guid": ""
            },
            "wavelengths_range": {
                "w_start": 486.0,
                "w_end": 655.0,
                "w_sampling": 13
            },
            "consider_diffraction_effects": true,
            "acquisition_lag_time": 0.0,
            "transmittance_spectrum_guid": ""
        },
        "distortion_file_uri": "/app/assets/CameraInputFiles/diffractive_effects.OPTDistortion",
        "horz_pixel": 640,
        "vert_pixel": 480,
        "width": 5.0,
        "height": 5.0
    },
    "description": "",
    "metadata": {}
}


## Delete

Once the data is deleted from the server, you can still work with local data and maybe commit
later.

In [16]:
sensor2.delete()
print(sensor2)

local: {
    "name": "Camera_Parameter",
    "display_name": "",
    "description": "",
    "metadata": {},
    "sensor_guid": "",
    "result_file_name": "",
    "sensor": {
        "name": "Camera_Parameter",
        "camera": {
            "mode_photometric": {
                "acquisition_integration": 0.01,
                "gamma_correction": 2.2,
                "png_bits": "PNG_BITS_16",
                "mode_color": {
                    "white_balance_mode_none": {},
                    "red_spectrum_guid": "",
                    "green_spectrum_guid": "",
                    "blue_spectrum_guid": ""
                },
                "wavelengths_range": {
                    "w_start": 400.0,
                    "w_end": 700.0,
                    "w_sampling": 13
                },
                "consider_diffraction_effects": false,
                "acquisition_lag_time": 0.0,
                "transmittance_spectrum_guid": ""
            },
            "distortion_file_

In [17]:
sensor1.delete()

## Other sensors

### Irradiance sensor

In [18]:
sensor3 = p.create_sensor(name="Irradiance.1", feature_type=SensorIrradiance)
sensor3.commit()
print(sensor3)

{
    "name": "Irradiance.1",
    "metadata": {
        "UniqueId": "b9518390-c2ac-491c-aa43-43b194942ea2"
    },
    "sensor_guid": "4fc538bc-ea22-46ca-93f8-d18b296a42b6",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "Irradiance.1",
        "irradiance": {
            "mode_photometric": {},
            "integration_type": "INTEGRATION_TYPE_PLANAR",
            "dimensions": {
                "x_start": -50.0,
                "x_end": 50.0,
                "x_sampling": 100,
                "y_start": -50.0,
                "y_end": 50.0,
                "y_sampling": 100
            },
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
            ],
            "layer_type_none": {},
            "

In [19]:
sensor3.set_type_radiometric()
sensor3.set_layer_type_polarization()
sensor3.commit()
print(sensor3)

{
    "name": "Irradiance.1",
    "metadata": {
        "UniqueId": "b9518390-c2ac-491c-aa43-43b194942ea2"
    },
    "sensor_guid": "4fc538bc-ea22-46ca-93f8-d18b296a42b6",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "Irradiance.1",
        "irradiance": {
            "mode_radiometric": {},
            "integration_type": "INTEGRATION_TYPE_PLANAR",
            "dimensions": {
                "x_start": -50.0,
                "x_end": 50.0,
                "x_sampling": 100,
                "y_start": -50.0,
                "y_end": 50.0,
                "y_sampling": 100
            },
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
            ],
            "layer_type_polarization": {},
     

In [20]:
sensor3.delete()

### radiance sensor

In [21]:
sensor4 = p.create_sensor(name="Radiance.1", feature_type=SensorRadiance)
sensor4.commit()
print(sensor4)

{
    "name": "Radiance.1",
    "metadata": {
        "UniqueId": "38984c06-41d6-4abc-a012-1b1099d4b012"
    },
    "sensor_guid": "67009e72-13aa-4f67-98c9-1b4f071851c9",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "Radiance.1",
        "radiance": {
            "mode_photometric": {},
            "focal": 250.0,
            "integration_angle": 5.0,
            "dimensions": {
                "x_start": -50.0,
                "x_end": 50.0,
                "x_sampling": 100,
                "y_start": -50.0,
                "y_end": 50.0,
                "y_sampling": 100
            },
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
            ],
            "layer_type_none": {},
            

In [22]:
sensor4.focal = 200
sensor4.set_type_spectral()
sensor4.set_layer_type_source()
sensor4.commit()
print(sensor4)

{
    "name": "Radiance.1",
    "metadata": {
        "UniqueId": "38984c06-41d6-4abc-a012-1b1099d4b012"
    },
    "sensor_guid": "67009e72-13aa-4f67-98c9-1b4f071851c9",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "Radiance.1",
        "radiance": {
            "mode_spectral": {
                "wavelengths_range": {
                    "w_start": 400.0,
                    "w_end": 700.0,
                    "w_sampling": 13
                }
            },
            "focal": 200.0,
            "integration_angle": 5.0,
            "dimensions": {
                "x_start": -50.0,
                "x_end": 50.0,
                "x_sampling": 100,
                "y_start": -50.0,
                "y_end": 50.0,
                "y_sampling": 100
            },
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
         

In [23]:
sensor4.delete()
print(sensor4)

local: {
    "name": "Radiance.1",
    "display_name": "",
    "description": "",
    "metadata": {},
    "sensor_guid": "",
    "result_file_name": "",
    "sensor": {
        "name": "Radiance.1",
        "radiance": {
            "mode_spectral": {
                "wavelengths_range": {
                    "w_start": 400.0,
                    "w_end": 700.0,
                    "w_sampling": 13
                }
            },
            "focal": 200.0,
            "integration_angle": 5.0,
            "dimensions": {
                "x_start": -50.0,
                "x_end": 50.0,
                "x_sampling": 100,
                "y_start": -50.0,
                "y_end": 50.0,
                "y_sampling": 100
            },
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.

### 3D irradiance sensor

In [24]:
create_helper_geometries(p)
sensor5 = p.create_sensor(name="3D_Irradiance.2", feature_type=Sensor3DIrradiance)
face = p.find(name="TheBodyB/TheFaceF", feature_type=Face)[0]
sensor5.geometries = [face]
sensor5.commit()
print(sensor5)

{
    "name": "3D_Irradiance.2",
    "metadata": {
        "UniqueId": "b08ea519-7a5b-4616-986d-8679f4811c99"
    },
    "sensor_guid": "3bb5cfb9-98a2-4608-86fa-783cfbf0d996",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "3D_Irradiance.2",
        "irradiance_3d": {
            "mode_photometric": {},
            "integration_type": "INTEGRATION_TYPE_PLANAR",
            "reflection": true,
            "transmission": true,
            "absorption": true,
            "layer_type_none": {},
            "geometries": {
                "geo_paths": [
                    "TheBodyB/TheFaceF"
                ]
            },
            "ray_file_type": "RayFileNone"
        },
        "description": "",
        "metadata": {}
    }
}


### Immersive sensor

An immersive sensor wraps the observer in a virtual cube and records light arriving from all
six directions (front, back, left, right, top, bottom).  It can be used with both direct and
inverse simulations.


Create an immersive sensor and commit it to show its default settings.

In [25]:
sensor6 = p.create_sensor(name="Immersive.1", feature_type=SensorImmersive)
print(sensor6)  # local: not yet on server

local: {
    "name": "Immersive.1",
    "display_name": "",
    "description": "",
    "metadata": {},
    "sensor_guid": "",
    "result_file_name": "",
    "sensor": {
        "name": "Immersive.1",
        "immersive": {
            "sampling": 600,
            "integration_angle": 5.0,
            "wavelengths_range": {
                "w_start": 400.0,
                "w_end": 700.0,
                "w_sampling": 13
            },
            "exclude_faces": {
                "front": false,
                "back": false,
                "left": false,
                "right": false,
                "top": false,
                "bottom": false
            },
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
            ],
            "layer_type_none": 

In [26]:
sensor6.commit()
print(sensor6)  # now on server

{
    "name": "Immersive.1",
    "metadata": {
        "UniqueId": "b99b4ec4-61ee-431b-96cd-25731745b617"
    },
    "sensor_guid": "2cc00c96-5f84-4db7-bf1b-cd9dbc7ec26a",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "Immersive.1",
        "immersive": {
            "sampling": 600,
            "integration_angle": 5.0,
            "wavelengths_range": {
                "w_start": 400.0,
                "w_end": 700.0,
                "w_sampling": 13
            },
            "exclude_faces": {
                "front": false,
                "back": false,
                "left": false,
                "right": false,
                "top": false,
                "bottom": false
            },
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
               

**Customise the sensor**

The sampling controls the pixel count per cube face, the integration angle is used for
direct simulations, and the wavelengths range selects the spectral window.

In [27]:
sensor6.sampling = 256
sensor6.integration_angle = 10.0
sensor6.stereo_interocular_distance = 50

In [28]:
wl = sensor6.set_wavelengths_range()
wl.start = 380.0
wl.end = 780.0
wl.sampling = 20

In [29]:
# Exclude the bottom face so rays from below are not recorded.
sensor6.exclude_bottom = True

In [30]:
# Separate results by light source.
sensor6.set_layer_type_source()

In [31]:
# Reposition the sensor (Origin, X-axis, Y-axis, Z-axis).
sensor6.axis_system = [5, 0, 10, 1, 0, 0, 0, 1, 0, 0, 0, 1]

In [32]:
sensor6.commit()
print(sensor6)

{
    "name": "Immersive.1",
    "metadata": {
        "UniqueId": "b99b4ec4-61ee-431b-96cd-25731745b617"
    },
    "sensor_guid": "2cc00c96-5f84-4db7-bf1b-cd9dbc7ec26a",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "Immersive.1",
        "immersive": {
            "sampling": 256,
            "stereo": {
                "interocular_distance": 50.0
            },
            "integration_angle": 10.0,
            "wavelengths_range": {
                "w_start": 380.0,
                "w_end": 780.0,
                "w_sampling": 20
            },
            "exclude_faces": {
                "bottom": true,
                "front": false,
                "back": false,
                "left": false,
                "right": false,
                "top": false
            },
            "axis_system": [
                5.0,
                0.0,
                10.0,
                1.0,
                0.0,
              

**Reset and delete**

In [33]:
sensor6.sampling = 512  # local modification — not yet committed
sensor6.reset()  # restores the last committed value (256)
print(sensor6)

{
    "name": "Immersive.1",
    "metadata": {
        "UniqueId": "b99b4ec4-61ee-431b-96cd-25731745b617"
    },
    "sensor_guid": "2cc00c96-5f84-4db7-bf1b-cd9dbc7ec26a",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "Immersive.1",
        "immersive": {
            "sampling": 256,
            "stereo": {
                "interocular_distance": 50.0
            },
            "integration_angle": 10.0,
            "wavelengths_range": {
                "w_start": 380.0,
                "w_end": 780.0,
                "w_sampling": 20
            },
            "exclude_faces": {
                "bottom": true,
                "front": false,
                "back": false,
                "left": false,
                "right": false,
                "top": false
            },
            "axis_system": [
                5.0,
                0.0,
                10.0,
                1.0,
                0.0,
              

In [34]:
sensor6.delete()
print(sensor6)

local: {
    "name": "Immersive.1",
    "display_name": "",
    "description": "",
    "metadata": {},
    "sensor_guid": "",
    "result_file_name": "",
    "sensor": {
        "name": "Immersive.1",
        "immersive": {
            "sampling": 256,
            "stereo": {
                "interocular_distance": 50.0
            },
            "integration_angle": 10.0,
            "wavelengths_range": {
                "w_start": 380.0,
                "w_end": 780.0,
                "w_sampling": 20
            },
            "exclude_faces": {
                "bottom": true,
                "front": false,
                "back": false,
                "left": false,
                "right": false,
                "top": false
            },
            "axis_system": [
                5.0,
                0.0,
                10.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
  

### Observer sensor

An observer sensor places multiple virtual viewpoints on a sphere around the scene.
It is useful when you want to sample a setup from several directions with one feature.

**Default values**

In [35]:
sensor7 = p.create_sensor(name="Observer.1", feature_type=SensorObserver)
print(sensor7)  # local: not yet on server

local: {
    "name": "Observer.1",
    "display_name": "",
    "description": "",
    "metadata": {},
    "sensor_guid": "",
    "result_file_name": "",
    "sensor": {
        "name": "Observer.1",
        "observer": {
            "focal": 50.0,
            "integration_angle": 5.0,
            "wavelengths_range": {
                "w_start": 400.0,
                "w_end": 700.0,
                "w_sampling": 13
            },
            "dimensions": {
                "x_start": -50.0,
                "x_end": 50.0,
                "x_sampling": 100,
                "y_start": -50.0,
                "y_end": 50.0,
                "y_sampling": 100
            },
            "distance": 1000.0,
            "sensors_locations": {
                "x_start": -45.0,
                "x_end": 45.0,
                "x_sampling": 5,
                "y_start": -30.0,
                "y_end": 30.0,
                "y_sampling": 3
            },
            "axis_system": [
                0

In [36]:
sensor7.commit()
print(sensor7)  # now on server

{
    "name": "Observer.1",
    "metadata": {
        "UniqueId": "7c677570-ba57-4864-abf8-d66a11852795"
    },
    "sensor_guid": "768489f4-f6ba-407b-aaee-ac8dcfcde201",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "Observer.1",
        "observer": {
            "focal": 50.0,
            "integration_angle": 5.0,
            "wavelengths_range": {
                "w_start": 400.0,
                "w_end": 700.0,
                "w_sampling": 13
            },
            "dimensions": {
                "x_start": -50.0,
                "x_end": 50.0,
                "x_sampling": 100,
                "y_start": -50.0,
                "y_end": 50.0,
                "y_sampling": 100
            },
            "distance": 1000.0,
            "sensors_locations": {
                "x_start": -45.0,
                "x_end": 45.0,
                "x_sampling": 5,
                "y_start": -30.0,
                "y_end": 30.0,


**Customise the sensor**

The focal distance defines the observer plane, ``distance`` controls the radius of the
sampling sphere, and the angular range controls how many viewpoints are created.

In [37]:
sensor7.focal = 320.0
sensor7.integration_angle = 7.5
sensor7.distance = 130.0
sensor7.stereo_interocular_distance = 63.0

In [38]:
observer_wl = sensor7.set_wavelengths_range()
observer_wl.start = 430.0
observer_wl.end = 670.0
observer_wl.sampling = 18

In [39]:
observer_dims = sensor7.set_dimensions()
observer_dims.x_start = -90.0
observer_dims.x_end = 90.0
observer_dims.x_sampling = 110
observer_dims.y_start = -70.0
observer_dims.y_end = 70.0
observer_dims.y_sampling = 75

In [40]:
observer_angles = sensor7.set_angular_range()
observer_angles.x_start = -55.0
observer_angles.x_end = 55.0
observer_angles.x_sampling = 9
observer_angles.y_start = -42.0
observer_angles.y_end = 42.0
observer_angles.y_sampling = 6

In [41]:
# Separate the recorded results by source and move the sensor frame.
sensor7.set_layer_type_source()
sensor7.axis_system = [10, 0, 15, 1, 0, 0, 0, 1, 0, 0, 0, 1]

In [42]:
sensor7.commit()
print(sensor7)

{
    "name": "Observer.1",
    "metadata": {
        "UniqueId": "7c677570-ba57-4864-abf8-d66a11852795"
    },
    "sensor_guid": "768489f4-f6ba-407b-aaee-ac8dcfcde201",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "Observer.1",
        "observer": {
            "focal": 320.0,
            "integration_angle": 7.5,
            "wavelengths_range": {
                "w_start": 430.0,
                "w_end": 670.0,
                "w_sampling": 18
            },
            "dimensions": {
                "x_start": -90.0,
                "x_end": 90.0,
                "x_sampling": 110,
                "y_start": -70.0,
                "y_end": 70.0,
                "y_sampling": 75
            },
            "stereo": {
                "interocular_distance": 63.0
            },
            "distance": 130.0,
            "sensors_locations": {
                "x_start": -55.0,
                "x_end": 55.0,
               

**Create from parameter dataclass**

Observer sensors can also be configured in one shot with a parameter dataclass.

In [43]:
observer_params = ObserverSensorParameters(
    focal=280.0,
    integration_angle=6.0,
    distance=110.0,
    interocular_distance=66.0,
    axis_system=[5, 10, 15, 1, 0, 0, 0, 1, 0, 0, 0, 1],
    layer_type="by_source",
)
sensor8 = p.create_sensor(
    name="Observer.Parameters",
    feature_type=SensorObserver,
    parameters=observer_params,
)
sensor8.commit()
print(sensor8)

{
    "name": "Observer.Parameters",
    "metadata": {
        "UniqueId": "8ab1afe7-b060-4427-a7be-866901190f98"
    },
    "sensor_guid": "41aeefab-c7ef-4a88-94b6-adbc2947dedd",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "Observer.Parameters",
        "observer": {
            "focal": 280.0,
            "integration_angle": 6.0,
            "wavelengths_range": {
                "w_start": 400.0,
                "w_end": 700.0,
                "w_sampling": 13
            },
            "dimensions": {
                "x_start": -50.0,
                "x_end": 50.0,
                "x_sampling": 100,
                "y_start": -50.0,
                "y_end": 50.0,
                "y_sampling": 100
            },
            "stereo": {
                "interocular_distance": 66.0
            },
            "distance": 110.0,
            "sensors_locations": {
                "x_start": -45.0,
                "x_end": 45

**Reset and delete**

In [44]:
sensor7.distance = 300.0  # local modification — not yet committed
sensor7.stereo_interocular_distance = None
sensor7.reset()  # restores the last committed values from the server
print(sensor7)

{
    "name": "Observer.1",
    "metadata": {
        "UniqueId": "7c677570-ba57-4864-abf8-d66a11852795"
    },
    "sensor_guid": "768489f4-f6ba-407b-aaee-ac8dcfcde201",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "Observer.1",
        "observer": {
            "focal": 320.0,
            "integration_angle": 7.5,
            "wavelengths_range": {
                "w_start": 430.0,
                "w_end": 670.0,
                "w_sampling": 18
            },
            "dimensions": {
                "x_start": -90.0,
                "x_end": 90.0,
                "x_sampling": 110,
                "y_start": -70.0,
                "y_end": 70.0,
                "y_sampling": 75
            },
            "stereo": {
                "interocular_distance": 63.0
            },
            "distance": 130.0,
            "sensors_locations": {
                "x_start": -55.0,
                "x_end": 55.0,
               

In [45]:
sensor8.delete()
print(sensor8)

local: {
    "name": "Observer.Parameters",
    "display_name": "",
    "description": "",
    "metadata": {},
    "sensor_guid": "",
    "result_file_name": "",
    "sensor": {
        "name": "Observer.Parameters",
        "observer": {
            "focal": 280.0,
            "integration_angle": 6.0,
            "wavelengths_range": {
                "w_start": 400.0,
                "w_end": 700.0,
                "w_sampling": 13
            },
            "dimensions": {
                "x_start": -50.0,
                "x_end": 50.0,
                "x_sampling": 100,
                "y_start": -50.0,
                "y_end": 50.0,
                "y_sampling": 100
            },
            "stereo": {
                "interocular_distance": 66.0
            },
            "distance": 110.0,
            "sensors_locations": {
                "x_start": -45.0,
                "x_end": 45.0,
                "x_sampling": 5,
                "y_start": -30.0,
                "y_end

In [46]:
sensor7.delete()
print(sensor7)

local: {
    "name": "Observer.1",
    "display_name": "",
    "description": "",
    "metadata": {},
    "sensor_guid": "",
    "result_file_name": "",
    "sensor": {
        "name": "Observer.1",
        "observer": {
            "focal": 320.0,
            "integration_angle": 7.5,
            "wavelengths_range": {
                "w_start": 430.0,
                "w_end": 670.0,
                "w_sampling": 18
            },
            "dimensions": {
                "x_start": -90.0,
                "x_end": 90.0,
                "x_sampling": 110,
                "y_start": -70.0,
                "y_end": 70.0,
                "y_sampling": 75
            },
            "stereo": {
                "interocular_distance": 63.0
            },
            "distance": 130.0,
            "sensors_locations": {
                "x_start": -55.0,
                "x_end": 55.0,
                "x_sampling": 9,
                "y_start": -42.0,
                "y_end": 42.0,
          

### Human eye sensor

A human eye sensor simulates what a human eye perceives when looking from an observer point
to a target point. The spectral range is fixed from 360 nm to 830 nm; only the wavelength
sampling can be changed. This sensor requires a Speos server 2027 R1 or above.

**Default values**

Colorimetric, wavelength sampling 13, pupil diameter 4 mm, horizontal field of view +/- 20 deg
(sampling 2352), vertical field of view +/- 10 deg (sampling 1176), no layer separation,
origin axis system and no focal (the target point is not defined yet).

In [47]:
if server_version_checker.is_version_supported(2027, 1, 0):
    sensor_eye = p.create_sensor(name="HumanEye.1", feature_type=SensorHumanEye)
    print(sensor_eye)  # local: not yet on server

    # The focal must be defined before committing (here by giving the target point)
    sensor_eye.target_point = [0, 0, 100]
    sensor_eye.commit()
    print(sensor_eye)  # now on server

    # **Customise the sensor**
    #
    # The sensor is positioned with an observer point, a target point and a top direction.
    # The focal length is the distance between the observer and the target points.

    sensor_eye.set_type_spectral()
    sensor_eye.wavelength_sampling = 25
    sensor_eye.pupil_diameter = 6.0
    eye_fov = sensor_eye.set_dimensions()
    eye_fov.x_sampling = 1200
    eye_fov.y_sampling = 600
    sensor_eye.observer_point = [0, 0, 0]
    sensor_eye.target_point = [0, 0, 500]
    sensor_eye.top_direction = [0, 1, 0]
    sensor_eye.set_layer_type_source()
    sensor_eye.commit()
    print(sensor_eye)
    print(sensor_eye.focal, sensor_eye.target_point)

    # **Create from parameter dataclass**
    #
    # The dataclass defines the axis system (observer, top direction Y, sight direction Z)
    # and the focal length.

    eye_params = HumanEyeSensorParameters(
        sensor_type=HumanEyeSensorTypes.colorimetric,
        pupil_diameter=5.0,
        axis_system=[0, 50, -200, 1, 0, 0, 0, 1, 0, 0, 0, 1],
        focal=200.0,
    )
    sensor_eye2 = p.create_sensor(
        name="HumanEye.Parameters", feature_type=SensorHumanEye, parameters=eye_params
    )
    sensor_eye2.commit()
    print(sensor_eye2)

    sensor_eye2.delete()
    sensor_eye.delete()

local: {
    "name": "HumanEye.1",
    "display_name": "",
    "description": "",
    "metadata": {},
    "sensor_guid": "",
    "result_file_name": "",
    "sensor": {
        "name": "HumanEye.1",
        "human_eye": {
            "mode_colorimetric": {
                "wavelengths_range": {
                    "w_start": 360.0,
                    "w_end": 830.0,
                    "w_sampling": 13
                }
            },
            "pupil_diameter": 4.0,
            "dimensions": {
                "x_start": -20.0,
                "x_end": 20.0,
                "x_sampling": 2352,
                "y_start": -10.0,
                "y_end": 10.0,
                "y_sampling": 1176
            },
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
 

{
    "name": "HumanEye.Parameters",
    "metadata": {
        "UniqueId": "5763e619-08c4-442b-9048-7548a909010b"
    },
    "sensor_guid": "b33b1020-8e11-496b-b2ca-0712509b774a",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "HumanEye.Parameters",
        "human_eye": {
            "mode_colorimetric": {
                "wavelengths_range": {
                    "w_start": 360.0,
                    "w_end": 830.0,
                    "w_sampling": 13
                }
            },
            "pupil_diameter": 5.0,
            "dimensions": {
                "x_start": -20.0,
                "x_end": 20.0,
                "x_sampling": 2352,
                "y_start": -10.0,
                "y_end": 10.0,
                "y_sampling": 1176
            },
            "axis_system": [
                0.0,
                50.0,
                -200.0,
                1.0,
                0.0,
                0.0,
           

### Polar intensity sensor

A polar intensity sensor generates an IES/Eulumdat photometric file from the simulation.
It supports format selection, explicit angular sampling, and far-field or near-field setup.

In [48]:
sensor_polar = p.create_sensor(name="PolarIntensity.1", feature_type=SensorPolarIntensity)
print(sensor_polar)  # local: not yet on server

local: {
    "name": "PolarIntensity.1",
    "display_name": "",
    "description": "",
    "metadata": {},
    "sensor_guid": "",
    "result_file_name": "",
    "sensor": {
        "name": "PolarIntensity.1",
        "polar_intensity": {
            "result_format": "FORMAT_IESNA_C",
            "dimensions": {
                "horizontal_sampling": 720,
                "vertical_sampling": 361
            },
            "far_field": {
                "integration_angle": 1.0
            },
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
            ]
        },
        "description": "",
        "metadata": {}
    }
}


In [49]:
sensor_polar.commit()
print(sensor_polar)  # now on server

{
    "name": "PolarIntensity.1",
    "metadata": {
        "UniqueId": "95dd6d2f-1798-40fe-9a6b-ef659a815fee"
    },
    "sensor_guid": "43b791c3-b3d3-4316-b84f-9bff1925dc68",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "PolarIntensity.1",
        "polar_intensity": {
            "result_format": "FORMAT_IESNA_C",
            "dimensions": {
                "horizontal_sampling": 720,
                "vertical_sampling": 361
            },
            "far_field": {
                "integration_angle": 1.0
            },
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
            ]
        },
        "description": "",
        "metadata": {}
    }
}


In [50]:
# Customise format, sampling and field configuration.
assert isinstance(sensor_polar, SensorPolarIntensity)
sensor_polar.set_format_eulumdat()
sensor_polar.set_constant_sampling()
sensor_polar.horizontal_sampling = 180
sensor_polar.vertical_sampling = 90
sensor_polar.set_far_field()
sensor_polar.integration_angle = 1
sensor_polar.axis_system = [0, 0, 25, 1, 0, 0, 0, 1, 0, 0, 0, 1]

In [51]:
sensor_polar.commit()
print(sensor_polar)

{
    "name": "PolarIntensity.1",
    "metadata": {
        "UniqueId": "95dd6d2f-1798-40fe-9a6b-ef659a815fee"
    },
    "sensor_guid": "43b791c3-b3d3-4316-b84f-9bff1925dc68",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "PolarIntensity.1",
        "polar_intensity": {
            "result_format": "FORMAT_EULUMDAT",
            "dimensions": {
                "horizontal_sampling": 180,
                "vertical_sampling": 90
            },
            "far_field": {
                "integration_angle": 1.0
            },
            "axis_system": [
                0.0,
                0.0,
                25.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
            ]
        },
        "description": "",
        "metadata": {}
    }
}


In [52]:
# The same sensor can be created directly from parameter dataclasses.
polar_param = PolarIntensitySensorParameters(
    format=PolarIntensityFormatTypes.iesna_b,
    dimensions=PolarIntensityDimensionsParameters(horizontal_sampling=37, vertical_sampling=37),
    integration_angle=1.0,
    axis_system=[10, 0, 30, 1, 0, 0, 0, 1, 0, 0, 0, 1],
)

In [53]:
sensor_polar_param = p.create_sensor(
    name="PolarIntensity.Parameters",
    feature_type=SensorPolarIntensity,
    parameters=polar_param,
)
sensor_polar_param.commit()
print(sensor_polar_param)

{
    "name": "PolarIntensity.Parameters",
    "metadata": {
        "UniqueId": "0384aa3f-db0c-4e43-8b7b-05d172efe49d"
    },
    "sensor_guid": "127bdce4-16d5-4cff-a380-59cc0540cce8",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "PolarIntensity.Parameters",
        "polar_intensity": {
            "result_format": "FORMAT_IESNA_B",
            "dimensions": {
                "horizontal_sampling": 37,
                "vertical_sampling": 37
            },
            "far_field": {
                "integration_angle": 1.0
            },
            "axis_system": [
                10.0,
                0.0,
                30.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
            ]
        },
        "description": "",
        "metadata": {}
    }
}


In [54]:
sensor_polar_param.delete()
sensor_polar.delete()

### 3D energy density sensor (Speos 2027 R1 or later)

This sensor records energy density inside a volume, rather than on a surface.
The default is a photometric 50 mm cube centered on the axis-system origin,
with 100 voxels along each edge and no layer separation.
Only photometric and radiometric modes and none/source/face layers are supported.
Light Expert and result-file readers/VTP exporters are not supported.

In [55]:
if server_version_checker.is_version_supported(2027, 1, 0):
    energy = p.create_sensor(name="EnergyDensity.1", feature_type=Sensor3DEnergyDensity)
    print(energy)
    energy.commit()
    print(energy)

    energy.set_type_radiometric().set_layer_type_source()
    volume = energy.set_dimensions()
    volume.x_size = 60.0
    volume.y_size = 40.0
    volume.z_size = 20.0
    volume.x_sampling = 60
    volume.y_sampling = 40
    volume.z_sampling = 20
    energy.axis_system = [10, 0, 25, 1, 0, 0, 0, 1, 0, 0, 0, 1]
    energy.result_file_name = "EnergyDensityVolume"
    energy.commit()
    print(energy)

    # The same volume can be configured through parameter dataclasses.
    energy_parameters = EnergyDensity3DSensorParameters(
        dimensions=EnergyDensity3DDimensionsParameters(
            x_size=60,
            x_sampling=60,
            y_size=40,
            y_sampling=40,
            z_size=20,
            z_sampling=20,
        ),
        sensor_type=SensorTypes.radiometric,
        layer_type=LayerTypes.by_source,
    )
    energy_from_parameters = p.create_sensor(
        name="EnergyDensity.Parameters",
        feature_type=Sensor3DEnergyDensity,
        parameters=energy_parameters,
    ).commit()
    print(energy_from_parameters)

    # With the graphics extra installed, p.preview() includes the oriented volume.
    # Export a simulation containing the sensor to save it in a .speos project;
    # lightbox exports do not retain sensors. Speos export also omits template metadata.
    energy.dimensions.z_size = 100
    energy.reset()
    print(energy)
    energy_from_parameters.delete()
    energy.delete()

local: {
    "name": "EnergyDensity.1",
    "display_name": "",
    "description": "",
    "metadata": {},
    "sensor_guid": "",
    "result_file_name": "",
    "sensor": {
        "name": "EnergyDensity.1",
        "energy_density": {
            "mode_photometric": {},
            "dimensions_3d": {
                "x_size": 50.0,
                "x_sampling": 100,
                "y_size": 50.0,
                "y_sampling": 100,
                "z_size": 50.0,
                "z_sampling": 100
            },
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
            ],
            "layer_type_none": {}
        },
        "description": "",
        "metadata": {}
    }
}
{
    "name": "EnergyDensity.1",
    "metadata": {
        "UniqueId": "a856fe73-f1

{
    "name": "EnergyDensity.1",
    "metadata": {
        "UniqueId": "a856fe73-f1c3-4c53-9c14-a9ea6c2ea839"
    },
    "sensor_guid": "cba4ce22-7647-44db-b16d-2154121ea07e",
    "result_file_name": "EnergyDensityVolume",
    "display_name": "",
    "description": "",
    "sensor": {
        "name": "EnergyDensity.1",
        "energy_density": {
            "mode_radiometric": {},
            "dimensions_3d": {
                "x_size": 60.0,
                "x_sampling": 60,
                "y_size": 40.0,
                "y_sampling": 40,
                "z_size": 20.0,
                "z_sampling": 20
            },
            "axis_system": [
                10.0,
                0.0,
                25.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
            ],
            "layer_type_source": {}
        },
        "description": "",
 

{
    "name": "EnergyDensity.Parameters",
    "metadata": {
        "UniqueId": "56914d69-ffb0-4d84-a59d-9e49983c506e"
    },
    "sensor_guid": "3d888a87-bcfd-4488-b879-9f496f614b7c",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "EnergyDensity.Parameters",
        "energy_density": {
            "mode_radiometric": {},
            "dimensions_3d": {
                "x_size": 60.0,
                "x_sampling": 60,
                "y_size": 40.0,
                "y_sampling": 40,
                "z_size": 20.0,
                "z_sampling": 20
            },
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
            ],
            "layer_type_source": {}
        },
        "description": "",
    

In [56]:
speos.close()

True